In [10]:
import json
import jsonlines


In [11]:
def get_sc_data(sc_data_path):
    sc_data_list = []
    # 假设文件名为 'data.jsonl'
    is_exist = []
    with open(sc_data_path, 'r', encoding='utf-8') as file:
        for line in file:

            sc_data = json.loads(line)  # 解析json字符串为Python数据结构（dict、list等）
            q_id = sc_data['question_id']
            if q_id in is_exist: continue

            sc_data_list.append(sc_data)
            is_exist.append(q_id)
    sc_data_list = sorted(sc_data_list, key=lambda x: x["question_id"])
    return sc_data_list

In [12]:
data = get_sc_data("/mnt/intern/MXY/paper/rl/ours/CHESS_1/output_data/spider_chess_sc_train-1.jsonl")

In [13]:
data[-1]

{'db_id': 'restaurants',
 'db_schema': 'CREATE TABLE GEOGRAPHIC\n(\n\t"REGION" text, --  \n\tPRIMARY KEY ("CITY_NAME"),\n\t"CITY_NAME" text, --  \n);\n\nCREATE TABLE RESTAURANT\n(\n\t"NAME" text, --  \n\t"CITY_NAME" text, --  \n\tFOREIGN KEY ("CITY_NAME") REFERENCES `GEOGRAPHIC`("CITY_NAME"),\n\tPRIMARY KEY ("ID"),\n\t"ID" int, --  \n\t"RATING" real, --  \n);\n\nCREATE TABLE LOCATION\n(\n\tFOREIGN KEY ("CITY_NAME") REFERENCES `GEOGRAPHIC`("CITY_NAME"),\n\t"RESTAURANT_ID" int, --  \n\t"HOUSE_NUMBER" int, --  \n\tFOREIGN KEY ("RESTAURANT_ID") REFERENCES `RESTAURANT`("RESTAURANT_ID"),\n\tPRIMARY KEY ("RESTAURANT_ID"),\n\t"CITY_NAME" text, --  \n);',
 'question_id': 8658}

In [14]:
q_ids = []
for item in data:
    q_ids.append(item['question_id'])


In [15]:
def find_missing_numbers(nums, start=0, end=8658):
    full_set = set(range(start, end + 1))
    nums_set = set(nums)
    missing = sorted(full_set - nums_set)
    return missing

# 示例
nums = q_ids  # 假设这是你的列表
missing_numbers = find_missing_numbers(nums)
print("缺少的数字有：", missing_numbers)


缺少的数字有： []


In [8]:
len(missing_numbers)

774

In [29]:
len(data)

9428

## 替换失败的数据

In [19]:
fail_data_list = get_sc_data('/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/output_data/dev_sc_data_deepseek_chat_fail_data.jsonl')

In [34]:
sc_data_list = get_sc_data("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/output_data/train_full_schema.jsonl")

In [31]:
sc_data_list[0]

{'db_id': 'california_schools',
 'db_schema': "CREATE TABLE schools\n(\n\tFundingType TEXT null, -- Example Values: `Directly funded`, `Locally funded`, `Not in CS funding model`  | Column Description: Indicates the charter school funding type | Value Description: Values are as follows:  ·\xa0\xa0\xa0\xa0\xa0\xa0 Not in CS (California School) funding model  ·\xa0\xa0\xa0\xa0\xa0\xa0 Locally funded  ·\xa0\xa0\xa0\xa0\xa0\xa0 Directly funded\n\tAdmFName3 TEXT null, -- Example Values: `('Drew',)`, `('Irma',)`, `('Vickie',)`   \n\tGSoffered TEXT null, -- Example Values: `K-12` | Column Name Meaning: grade span offered | Column Description: The grade span offered is the lowest grade and the highest grade offered or supported by the school, district, or administrative authority. This field might differ from the grade span served as reported in the most recent certified California Longitudinal Pupil Achievement (CALPADS) Fall 1 data collection. | Value Description: For example XYZ School migh

In [7]:
for i in sc_data_list:
    for j in fail_data_list:
        if i['question_id'] == j['question_id']:
            print('替换前', i['db_schema'])
            i ['db_schema'] = j['db_schema']
            print('替换后', i['db_schema'])



NameError: name 'fail_data_list' is not defined

In [32]:
original_data_list = json.load(open("/mnt/lustre/maxingyu/paper/rl/ours/verl/data/oiginal_data/bird/train.json", 'r'))

In [35]:
len(sc_data_list), len(original_data_list)

(9428, 9428)

In [36]:
for sc_data, original_data in zip(sc_data_list, original_data_list):
    original_data['db_schema'] = sc_data['db_schema']


In [37]:
original_data_list[100]

{'db_id': 'movie_platform',
 'db_schema': 'CREATE TABLE lists_users\n(\n\tlist_id INTEGER not null, -- Example Values: `(192287,)`, `(192313,)`, `(192318,)`   \n\tlist_creation_date_utc TEXT, -- Example Values: `(\'2009-12-18\',)`, `(\'2010-01-30\',)`, `(\'2010-03-31\',)`   \n\tuser_trialist INTEGER, -- Example Values: `1`, `0`  | Column Description: whether the user was a tralist when he created the list | Value Description: 1 = the user was a trialist when he created the list  0 = the user was not a trialist when he created the list\n\tforeign key (list_id) references lists(list_id),\n\tlist_update_date_utc TEXT, -- Example Values: `(\'2019-11-26\',)`, `(\'2020-05-01\',)`, `(\'2020-04-12\',)`   \n\tuser_has_payment_method TEXT, -- Example Values: `1`, `0`  | Column Description: whether the user was a paying subscriber when he created the list | Value Description: 1 = the user was a paying subscriber when he created the list  0 = the user was not a paying subscriber when he created th

In [16]:
json.dump(original_data_list, open("/mnt/lustre/maxingyu/paper/rl/ours/verl/data/oiginal_data/bird_chess_full_schema/train.json", 'w'), indent=4, ensure_ascii=False)

In [1]:
print("CREATE TABLE frpm\n(\n\t`County Name` TEXT null, -- Example Values: `Alameda`   \n\t`Enrollment (K-12)` REAL null, -- Example Values: `(1087.0,)`, `(395.0,)`, `(244.0,)`  | Column Description: Enrollment (K-12) | Value Description: K-12: 1st grade - 12nd grade\n\t`Free Meal Count (K-12)` REAL null, -- Example Values: `(565.0,)`, `(186.0,)`, `(134.0,)`  | Column Description: Free Meal Count (K-12) | Value Description: eligible free rate = Free Meal Count / Enrollment\n\tCDSCode TEXT not null primary key,\n);")

CREATE TABLE frpm
(
	`County Name` TEXT null, -- Example Values: `Alameda`   
	`Enrollment (K-12)` REAL null, -- Example Values: `(1087.0,)`, `(395.0,)`, `(244.0,)`  | Column Description: Enrollment (K-12) | Value Description: K-12: 1st grade - 12nd grade
	`Free Meal Count (K-12)` REAL null, -- Example Values: `(565.0,)`, `(186.0,)`, `(134.0,)`  | Column Description: Free Meal Count (K-12) | Value Description: eligible free rate = Free Meal Count / Enrollment
	CDSCode TEXT not null primary key,
);


In [1]:
print("CREATE TABLE singer\n(\n\tPRIMARY KEY (\"Singer_ID\"),\n\t\"Country\" text, -- Example Values: `Netherlands`, `United States`, `France`, `UK`, `USA`   \n\t\"Age\" int, -- Example Values: `52`, `32`, `29`, `41`, `43`   \n\t\"Singer_ID\" int, --  \n);")

CREATE TABLE singer
(
	PRIMARY KEY ("Singer_ID"),
	"Country" text, -- Example Values: `Netherlands`, `United States`, `France`, `UK`, `USA`   
	"Age" int, -- Example Values: `52`, `32`, `29`, `41`, `43`   
	"Singer_ID" int, --  
);


## 处理spider数据评估指标

In [1]:
import json

In [5]:
data = json.load(open("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/results/test/CHESS_IR_SS_CG/test/2025-08-01T15:31:44.889175/-predictions.json", 'r'))

In [3]:
sqls = []
with open("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/results/test/CHESS_IR_SS_CG/test/2025-08-01T15:31:44.889175/-predictions.sql", 'w') as f:
    for k,v in data.items():
        sql = v.split("\t----- bird -----\t")[0]
        # sqls.append(sql)
        f.write(sql+'\n')

In [4]:
sqls[0]

IndexError: list index out of range

## 处理spider的db schema

In [17]:
sc_data_list = get_sc_data("/mnt/intern/MXY/paper/rl/ours/CHESS_1/output_data/spider_chess_sc_train-1.jsonl")
original_data_list = json.load(open("/mnt/intern/MXY/paper/rl/ours/CHESS_1/spider/train/train.json", 'r'))


In [18]:
len(sc_data_list), len(original_data_list)

(8659, 8659)

In [19]:
sc_data_list[0]

{'db_id': 'department_management',
 'db_schema': 'CREATE TABLE head\n(\n\t"age" real, -- Example Values: `67.0`, `68.0`, `69.0`, `52.0`, `53.0`   \n\t"head_ID" int, --  \n\tPRIMARY KEY ("head_ID"),\n);',
 'question_id': 0}

In [22]:
new_data = []
for i in original_data_list:
    flag = False
    for j in sc_data_list:
        if i['question_id'] == j['question_id']:
            new_data.append({
                'db_id': i['db_id'],
                'db_schema': j['db_schema'],
                'question': i['question'],
                'evidence': '',
                'gold_sql': i['SQL']
            })
            flag = True
            break
    if not flag:
        print('=====')
    

In [23]:
new_data[0]


{'db_id': 'department_management',
 'db_schema': 'CREATE TABLE head\n(\n\t"age" real, -- Example Values: `67.0`, `68.0`, `69.0`, `52.0`, `53.0`   \n\t"head_ID" int, --  \n\tPRIMARY KEY ("head_ID"),\n);',
 'question': 'How many heads of the departments are older than 56 ?',
 'evidence': '',
 'gold_sql': 'SELECT count(*) FROM head WHERE age > 56 ;'}

In [24]:
json.dump(new_data, open("/mnt/intern/MXY/paper/rl/noise_aware/verl/data/original_data/spider/v0_sc/data/train.json", 'w'), indent=4, ensure_ascii=False)


In [16]:
data = json.load(open("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/data/train/train.json", 'r'))

In [17]:
for idx, item in enumerate(data):
    item['question_id'] = idx

In [19]:
json.dump(data, open("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/data/train/train.json", 'w'), ensure_ascii=False, indent=4)

## 处理spider2的db schema

In [23]:
sc_data_list = get_sc_data("/mnt/lustre/maxingyu/paper/rl/ours/CHESS_1/output_data/spider2_dev_full_schema.jsonl")
original_data_list = json.load(open("/mnt/lustre/maxingyu/paper/rl/ours/verl/data/oiginal_data/eval_data/spider2_full_schema_dev.json", 'r'))


In [24]:
sc_data_list[0]

{'db_id': 'E_commerce',
 'db_schema': 'CREATE TABLE products\n(\n\t"product_id" TEXT, -- Example Values: `(\'1e9e8ef04dbcff4541ed26657ea517e5\',)`, `(\'3aa071139cb16b67ca9e5dea641aaa2f\',)`, `(\'96bd76ec8810374ed1b65e291975717f\',)`   \n\t"product_weight_g" REAL, -- Example Values: `(225.0,)`, `(1000.0,)`, `(154.0,)`   \n\t"product_name_lenght" REAL, -- Example Values: `(40.0,)`, `(44.0,)`, `(46.0,)`   \n\t"product_photos_qty" REAL, -- Example Values: `(1.0,)`, `(4.0,)`, `(2.0,)`   \n\t"product_length_cm" REAL, -- Example Values: `(16.0,)`, `(30.0,)`, `(18.0,)`   \n\t"product_height_cm" REAL, -- Example Values: `(10.0,)`, `(18.0,)`, `(9.0,)`   \n\t"product_width_cm" REAL, -- Example Values: `(14.0,)`, `(20.0,)`, `(15.0,)`   \n\t"product_category_name" TEXT, -- Example Values: `(\'perfumaria\',)`, `(\'artes\',)`, `(\'esporte_lazer\',)`   \n\t"product_description_lenght" REAL, -- Example Values: `(287.0,)`, `(276.0,)`, `(250.0,)`   \n);\n\nCREATE TABLE customers\n(\n\t"customer_id" TEXT,

In [20]:
print(original_data_list[0]['db_schema'])

CREATE TABLE `product_category_name_translation` (
  product_category_name TEXT, -- Example Values: `beleza_saude`, `informatica_acessorios` 
  product_category_name_english TEXT, -- Example Values: `health_beauty`, `computers_accessories` 
);


CREATE TABLE `sellers` (
  seller_id TEXT, -- Example Values: `3442f8959a84dea7ee197c632cb2df15`, `d1b65fc7debc3361ea86b5f14c68d2e2` 
  seller_zip_code_prefix INTEGER, -- Example Values: `13023`, `13844` 
  seller_city TEXT, -- Example Values: `campinas`, `mogi guacu` 
  seller_state TEXT, -- Example Values: `SP`, `SP` 
);


CREATE TABLE `customers` (
  customer_id TEXT, -- Example Values: `06b8999e2fba1a1fbc88172c00ba8bc7`, `18955e83d337fd6b2def6b18a428ac77` 
  customer_unique_id TEXT, -- Example Values: `861eff4711a542e4b93843c6dd7febb0`, `290c77bc529b7ac935b93aa66c333dc3` 
  customer_zip_code_prefix INTEGER, -- Example Values: `14409`, `9790` 
  customer_city TEXT, -- Example Values: `franca`, `sao bernardo do campo` 
  customer_state TEXT, 

In [22]:
print(original_data_list[0]['db_schema'])

CREATE TABLE products
(
	"product_id" TEXT, -- Example Values: `('1e9e8ef04dbcff4541ed26657ea517e5',)`, `('3aa071139cb16b67ca9e5dea641aaa2f',)`, `('96bd76ec8810374ed1b65e291975717f',)`   
	"product_weight_g" REAL, -- Example Values: `(225.0,)`, `(1000.0,)`, `(154.0,)`   
	"product_name_lenght" REAL, -- Example Values: `(40.0,)`, `(44.0,)`, `(46.0,)`   
	"product_photos_qty" REAL, -- Example Values: `(1.0,)`, `(4.0,)`, `(2.0,)`   
	"product_length_cm" REAL, -- Example Values: `(16.0,)`, `(30.0,)`, `(18.0,)`   
	"product_height_cm" REAL, -- Example Values: `(10.0,)`, `(18.0,)`, `(9.0,)`   
	"product_width_cm" REAL, -- Example Values: `(14.0,)`, `(20.0,)`, `(15.0,)`   
	"product_category_name" TEXT, -- Example Values: `('perfumaria',)`, `('artes',)`, `('esporte_lazer',)`   
	"product_description_lenght" REAL, -- Example Values: `(287.0,)`, `(276.0,)`, `(250.0,)`   
);

CREATE TABLE customers
(
	"customer_id" TEXT, -- Example Values: `('06b8999e2fba1a1fbc88172c00ba8bc7',)`, `('18955e83d337

In [25]:
for idx, item in enumerate(original_data_list):
    db_schema = item['db_schema']
    for j in sc_data_list:
        if idx == j['question_id']:
            # print("=======")
            db_schema = j['db_schema']
    item['db_schema'] = db_schema


In [17]:
original_data_list[0]

{'db_id': 'E_commerce',
 'instance_id': 'local002',
 'external_knowledge': None,
 'db_schema': 'CREATE TABLE products\n(\n\t"product_id" TEXT, -- Example Values: `(\'1e9e8ef04dbcff4541ed26657ea517e5\',)`, `(\'3aa071139cb16b67ca9e5dea641aaa2f\',)`, `(\'96bd76ec8810374ed1b65e291975717f\',)`   \n\t"product_weight_g" REAL, -- Example Values: `(225.0,)`, `(1000.0,)`, `(154.0,)`   \n\t"product_name_lenght" REAL, -- Example Values: `(40.0,)`, `(44.0,)`, `(46.0,)`   \n\t"product_photos_qty" REAL, -- Example Values: `(1.0,)`, `(4.0,)`, `(2.0,)`   \n\t"product_length_cm" REAL, -- Example Values: `(16.0,)`, `(30.0,)`, `(18.0,)`   \n\t"product_height_cm" REAL, -- Example Values: `(10.0,)`, `(18.0,)`, `(9.0,)`   \n\t"product_width_cm" REAL, -- Example Values: `(14.0,)`, `(20.0,)`, `(15.0,)`   \n\t"product_category_name" TEXT, -- Example Values: `(\'perfumaria\',)`, `(\'artes\',)`, `(\'esporte_lazer\',)`   \n\t"product_description_lenght" REAL, -- Example Values: `(287.0,)`, `(276.0,)`, `(250.0,)`  

In [27]:
json.dump(original_data_list, open("/mnt/lustre/maxingyu/paper/rl/ours/verl/data/oiginal_data/eval_data/spider2_chess_full_schema_dev.json", 'w'), ensure_ascii=False, indent=4)


In [16]:
import json 
